# Stress test de pareamento com Splink



Este notebook é uma versão única e documentada.

A ideia é testar várias formas de parear registros de pessoas usando o Splink e medir a performance com uma verdade conhecida.

Neste arquivo, a coluna `id_unico` funciona como o identificador real da pessoa. Ela será usada apenas para avaliação, não como variável de pareamento.

No notebook, a configuração feita está assim: link_type="dedupe_only". Isso significa: “Procure registros duplicados dentro de uma única tabela.” Então, mesmo tendo apenas um df, o Splink entende que cada linha pode ser uma pessoa registrada mais de uma vez. Caso houvesse mais de uma base, seria passado o parâmetro de forma diferente: link_only ou link_and_dedupe

### O que este notebook faz


1. Instala as bibliotecas necessárias.
2. Lê o arquivo `teste_pareamento.csv`.
3. Padroniza nomes, datas, CPF, CNS e fontes.
4. Cria variáveis auxiliares para pareamento.
5. Define cenários de blocking.
6. Define cenários de comparação.
7. Treina modelos Splink em diferentes configurações.
8. Mede precision, recall e F1.
9. Gera ranking dos melhores cenários.
10. Permite rodar o melhor cenário no arquivo completo.

### Como pensar o problema

Pareamento não é apenas procurar CPF igual.

Em bases reais de saúde, CPF, CNS, nome, data de nascimento e nomes dos pais podem estar incompletos, escritos de formas diferentes ou com erro de digitação.

Por isso, este notebook testa cenários diferentes:

- com CPF e CNS;
- sem CPF e CNS;
- com nome da mãe e nome do pai;
- sem nome dos pais;
- com regras de blocking mais restritas;
- com regras de blocking mais permissivas;
- com similaridade de nome mais rígida ou mais flexível.

# 1. Instalação



Esta célula instala o Splink e as bibliotecas usadas no notebook.

O Splink faz o pareamento probabilístico.
O DuckDB é usado como motor local de execução.
O pandas organiza as tabelas.
O numpy ajuda nos cálculos.
O pyarrow permite salvar arquivos em parquet.
O altair é usado por algumas visualizações do Splink.

Rode esta célula apenas uma vez no Colab. Se o ambiente reiniciar, rode de novo.

In [1]:
!pip install -U splink duckdb pandas numpy pyarrow altair --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 75.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0

# 2. Caminhos de entrada e saída



Aqui definimos:

- `csv_path`: caminho do CSV dentro do Colab.
- `out_dir`: pasta onde os resultados serão salvos.

Se o arquivo subir com outro nome, ajuste `csv_path`.

In [2]:
csv_path = '/content/teste_pareamento.csv'
out_dir = '/content/saida_splink'

print(csv_path)
print(out_dir)


/content/teste_pareamento.csv
/content/saida_splink


# 3. Funções de preparação dos dados



O Splink não deve receber os dados crus sem preparo. Antes do pareamento, fazemos uma padronização mínima.



### O que será criado



| Variável | Uso |
|---|---|
| `unique_id` | ID técnico da linha, usado pelo Splink |
| `source_dataset` | Fonte do registro, derivada de `fonte_dados` |
| `nome_paciente_std` | Nome do paciente limpo e sem acentos |
| `nome_mae_std` | Nome da mãe limpo e sem acentos |
| `nome_pai_std` | Nome do pai limpo e sem acentos |
| `primeiro_nome` | Primeiro token do nome do paciente |
| `ultimo_nome` | Último token do nome do paciente |
| `mae_primeiro_nome` | Primeiro token do nome da mãe |
| `mae_ultimo_nome` | Último token do nome da mãe |
| `dt_nasc_std` | Data de nascimento normalizada para `YYYY-MM-DD` |
| `ano_nasc`, `mes_nasc`, `dia_nasc` | Partes da data para regras alternativas |
| `cpf_digits` | CPF somente com números |
| `cns_digits` | CNS somente com números |
| `sexo_std` | Sexo padronizado de forma simples |


### Por que isso importa



Pequenas diferenças de escrita podem quebrar um pareamento determinístico.

Exemplo:

- `José da Silva`
- `JOSE SILVA`
- `Jose da S.`
- `JOSÉ  DA   SILVA`

Para o computador, isso pode parecer diferente. A padronização reduz parte desse ruído.

### Padronização e Normalização

In [7]:

# ============================================================
# Imports e funções de preparação
# ============================================================

from __future__ import annotations

import json
import math
import re
import time
import unicodedata
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl

try:
    from splink.blocking_analysis import count_comparisons_from_blocking_rule
    HAS_BLOCKING_ANALYSIS = True
except Exception:
    HAS_BLOCKING_ANALYSIS = False


def remover_acentos(texto: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def normalizar_texto(valor: Any) -> Optional[str]:
    if pd.isna(valor):
        return None

    s = str(valor).strip().lower()
    if not s:
        return None

    s = remover_acentos(s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s or None


def somente_digitos(valor: Any, min_len: int = 1) -> Optional[str]:
    if pd.isna(valor):
        return None

    s = re.sub(r"\D", "", str(valor))
    if len(s) < min_len:
        return None

    return s


def parse_data_brasil_ou_iso(valor: Any) -> Optional[str]:
    """
    Normaliza datas para YYYY-MM-DD.

    Tenta lidar com:
    - 2000-10-12
    - 17-03-2008
    - 03151938, possível MMDDYYYY
    - 15031938, possível DDMMYYYY
    """
    if pd.isna(valor):
        return None

    s = str(valor).strip()
    if not s:
        return None

    formatos = [
        "%Y-%m-%d",
        "%d-%m-%Y",
        "%m%d%Y",
        "%d%m%Y",
        "%Y%m%d",
        "%m-%d-%Y",
        "%d/%m/%Y",
        "%Y/%m/%d",
    ]

    for fmt in formatos:
        dt = pd.to_datetime(s, format=fmt, errors="coerce")
        if pd.notna(dt):
            return dt.strftime("%Y-%m-%d")

    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if pd.isna(dt):
        return None

    return dt.strftime("%Y-%m-%d")


def primeiro_token(s: Any) -> Optional[str]:
    s = normalizar_texto(s)
    if not s:
        return None
    partes = s.split()
    return partes[0] if partes else None


def ultimo_token(s: Any) -> Optional[str]:
    s = normalizar_texto(s)
    if not s:
        return None
    partes = s.split()
    return partes[-1] if partes else None


def preparar_df(caminho_csv: str | Path) -> pd.DataFrame:
    df = pd.read_csv(caminho_csv, dtype=str)

    cols_drop = [c for c in df.columns if c.lower().startswith("unnamed")]
    if cols_drop:
        df = df.drop(columns=cols_drop)

    colunas_esperadas = [
        "id_unico", "nome_paciente", "dt_nasc", "sexo", "CPF",
        "CNS", "nome_mae", "nome_pai", "fonte_dados"
    ]

    faltantes = [c for c in colunas_esperadas if c not in df.columns]
    if faltantes:
        raise ValueError(f"Colunas ausentes no CSV: {faltantes}")

    df = df.reset_index(drop=True).copy()

    # Identificador técnico da linha.
    # Este é o ID que o Splink usa para comparar registro_l x registro_r.
    df["unique_id"] = df.index.astype(str)

    # Fonte original.
    df["source_dataset"] = df["fonte_dados"].fillna("fonte_desconhecida").astype(str)

    # Ground truth.
    # Não usar em produção como variável de pareamento.
    # Aqui serve para medir precision, recall e F1.
    df["id_unico"] = df["id_unico"].astype(str)

    # Padronizações.
    df["nome_paciente_std"] = df["nome_paciente"].map(normalizar_texto)
    df["nome_mae_std"] = df["nome_mae"].map(normalizar_texto)
    df["nome_pai_std"] = df["nome_pai"].map(normalizar_texto)

    df["primeiro_nome"] = df["nome_paciente"].map(primeiro_token)
    df["ultimo_nome"] = df["nome_paciente"].map(ultimo_token)

    df["mae_primeiro_nome"] = df["nome_mae"].map(primeiro_token)
    df["mae_ultimo_nome"] = df["nome_mae"].map(ultimo_token)

    df["pai_primeiro_nome"] = df["nome_pai"].map(primeiro_token)
    df["pai_ultimo_nome"] = df["nome_pai"].map(ultimo_token)

    df["dt_nasc_std"] = df["dt_nasc"].map(parse_data_brasil_ou_iso)

    dt = pd.to_datetime(df["dt_nasc_std"], errors="coerce")
    df["ano_nasc"] = dt.dt.year.astype("Int64").astype(str).replace("<NA>", None)
    df["mes_nasc"] = dt.dt.month.astype("Int64").astype(str).replace("<NA>", None)
    df["dia_nasc"] = dt.dt.day.astype("Int64").astype(str).replace("<NA>", None)

    df["cpf_digits"] = df["CPF"].map(lambda x: somente_digitos(x, min_len=10))
    df["cns_digits"] = df["CNS"].map(lambda x: somente_digitos(x, min_len=10))

    df["sexo_std"] = (
        df["sexo"]
        .astype(str)
        .str.strip()
        .replace({"nan": None, "None": None, "": None})
    )

    return df


def amostrar_por_pessoa(df: pd.DataFrame, sample_records: int, seed: int = 42) -> pd.DataFrame:
    """
    Amostra por id_unico, preservando os clusters reais.
    Isso evita selecionar apenas uma linha isolada de uma pessoa.
    """
    if sample_records <= 0 or sample_records >= len(df):
        return df.copy()

    rng = np.random.default_rng(seed)
    ids = df["id_unico"].drop_duplicates().to_numpy()
    rng.shuffle(ids)

    escolhidos = []
    n = 0
    tamanhos = df["id_unico"].value_counts().to_dict()

    for id_ in ids:
        escolhidos.append(id_)
        n += tamanhos[id_]
        if n >= sample_records:
            break

    return df[df["id_unico"].isin(escolhidos)].copy().reset_index(drop=True)


def perfil_basico(df: pd.DataFrame) -> Dict[str, Any]:
    n = len(df)
    vc = df["id_unico"].value_counts()
    total_pairs = n * (n - 1) // 2
    true_pairs = int(((vc * (vc - 1)) // 2).sum())

    cross_true_pairs = 0
    same_source_true_pairs = 0

    for _, g in df.groupby("id_unico"):
        n_g = len(g)
        if n_g <= 1:
            continue

        total_g = n_g * (n_g - 1) // 2
        by_source = g["source_dataset"].value_counts()
        same_g = int(sum(v * (v - 1) // 2 for v in by_source))

        cross_true_pairs += total_g - same_g
        same_source_true_pairs += same_g

    return {
        "n_registros": n,
        "n_pessoas_reais": int(vc.shape[0]),
        "n_pessoas_com_duplicidade": int((vc > 1).sum()),
        "maior_cluster_real": int(vc.max()),
        "total_pares_possiveis": int(total_pairs),
        "total_pares_verdadeiros": int(true_pairs),
        "proporcao_pares_verdadeiros": true_pairs / total_pairs if total_pairs else None,
        "pares_verdadeiros_interfonte": int(cross_true_pairs),
        "pares_verdadeiros_intrafonte": int(same_source_true_pairs),
        "perc_pares_verdadeiros_interfonte": cross_true_pairs / true_pairs if true_pairs else None,
        "nulos_nome_mae": float(df["nome_mae_std"].isna().mean()),
        "nulos_nome_pai": float(df["nome_pai_std"].isna().mean()),
        "nulos_dt_nasc": float(df["dt_nasc_std"].isna().mean()),
        "nulos_cpf": float(df["cpf_digits"].isna().mean()),
        "nulos_cns": float(df["cns_digits"].isna().mean()),
    }


def mostrar_perfil(perfil: Dict[str, Any]) -> pd.DataFrame:
    return (
        pd.DataFrame({"indicador": list(perfil.keys()), "valor": list(perfil.values())})
        .assign(valor=lambda d: d["valor"].map(lambda x: round(x, 4) if isinstance(x, float) else x))
    )


# 4. Cenários de comparação e hiperparâmetros



Esta célula define os cenários que serão testados.



### Cenários de blocking


O blocking define quais pares serão comparados.

Sem blocking, compararíamos todos contra todos. Com 200 mil registros, isso geraria quase 20 bilhões de pares possíveis.

Isso é inviável e desnecessário.


Uma regra como:

```text
mesmo CPF
```

compara apenas registros que têm CPF igual.

Uma regra como:

```text
mesma data de nascimento + mesmo primeiro nome
```

compara registros que passam por essa condição.

Neste notebook, cada cenário de blocking representa uma hipótese diferente sobre a qualidade dos dados.


| Tipo de blocking | Vantagem | Risco |
|---|---|---|
| Muito restrito | Rápido e preciso | Perde pares verdadeiros |
| Muito amplo | Maior chance de achar pares | Gera muitos falsos candidatos e fica pesado |
| Balanceado | Melhor ponto inicial | Precisa testar |




| Cenário | Ideia |
|---|---|
| `ids_strict` | Usa CPF e CNS como principais chaves |
| `ids_plus_nome_data` | Usa CPF/CNS e também nome + data |
| `balanced` | Mistura identificadores, nome, data e nome da mãe |
| `no_ids_moderado` | Simula ausência de CPF/CNS |
| `broad_recall` | Cenário mais permissivo para tentar recuperar mais pares |



### Cenários de comparação



| Cenário | Ideia |
|---|---|
| `completo` | Usa todos os campos principais |
| `sem_ids` | Remove CPF/CNS das comparações |
| `sem_pais` | Remove nome da mãe e nome do pai |
| `nome_data_sexo` | Usa um modelo mais simples |



### Hiperparâmetros testados



O notebook varia principalmente:

- limiares de similaridade do nome do paciente;
- limiares de similaridade dos nomes dos pais;
- estratégia de treino;
- recall presumido das regras determinísticas;
- tamanho da amostra para estimar os pares não correspondentes.



### Interpretação prática



- Limiar de nome mais alto: exige nomes mais parecidos.
- Limiar de nome mais baixo: aceita mais variação, mas pode gerar falsos positivos.
- Com CPF/CNS: tende a ser mais preciso.
- Sem CPF/CNS: testa resiliência do modelo quando identificadores estão ruins ou ausentes.


### Funções de cenários

In [6]:

# ============================================================
# Configuração dos cenários Splink
# ============================================================

def regras_blocking(nome_cenario: str) -> List[Any]:
    """
    Mais regras tendem a aumentar recall, mas também aumentam custo computacional.
    """
    biblioteca = {
        "ids_strict": [
            block_on("cpf_digits"),
            block_on("cns_digits"),
        ],
        "ids_plus_nome_data": [
            block_on("cpf_digits"),
            block_on("cns_digits"),
            block_on("dt_nasc_std", "primeiro_nome"),
            block_on("dt_nasc_std", "ultimo_nome"),
        ],
        "balanced": [
            block_on("cpf_digits"),
            block_on("cns_digits"),
            block_on("dt_nasc_std", "primeiro_nome"),
            block_on("dt_nasc_std", "ultimo_nome"),
            block_on("primeiro_nome", "ultimo_nome"),
            block_on("mae_primeiro_nome", "dt_nasc_std"),
        ],
        "no_ids_moderado": [
            block_on("dt_nasc_std", "primeiro_nome"),
            block_on("dt_nasc_std", "ultimo_nome"),
            block_on("primeiro_nome", "ultimo_nome"),
            block_on("mae_primeiro_nome", "mae_ultimo_nome"),
            block_on("mae_primeiro_nome", "dt_nasc_std"),
        ],
        "broad_recall": [
            block_on("cpf_digits"),
            block_on("cns_digits"),
            block_on("dt_nasc_std", "primeiro_nome"),
            block_on("dt_nasc_std", "ultimo_nome"),
            block_on("primeiro_nome", "ultimo_nome"),
            block_on("mae_primeiro_nome", "dt_nasc_std"),
            block_on("pai_primeiro_nome", "dt_nasc_std"),
            block_on("substr(nome_paciente_std, 1, 1)", "ultimo_nome", "ano_nasc"),
        ],
    }

    if nome_cenario not in biblioteca:
        raise ValueError(f"Cenário de blocking desconhecido: {nome_cenario}")

    return biblioteca[nome_cenario]


def regras_deterministicas_para_prior() -> List[Any]:
    """
    Regras de alta precisão para estimar a probabilidade a priori
    de dois registros serem da mesma pessoa.
    """
    return [
        block_on("cpf_digits"),
        block_on("cns_digits"),
        block_on("nome_paciente_std", "dt_nasc_std"),
    ]


def regras_em_training() -> List[Any]:
    """
    Regras para EM.
    A ideia é variar campos de blocking para não impedir a estimação
    dos mesmos campos em todas as rodadas.
    """
    return [
        block_on("dt_nasc_std"),
        block_on("primeiro_nome", "ultimo_nome"),
        block_on("mae_primeiro_nome"),
    ]


@dataclass
class ExperimentConfig:
    nome: str
    blocking_scenario: str
    comparison_scenario: str
    name_thresholds: tuple
    parent_thresholds: tuple
    include_ids: bool
    include_parents: bool
    include_dob: bool
    include_sex: bool
    training_mode: str
    prior_recall: float
    u_max_pairs: float
    prediction_min_threshold: float
    seed: int = 42


def montar_comparisons(cfg: ExperimentConfig) -> List[Any]:
    comparisons = []

    # Nome do paciente: fuzzy matching.
    comparisons.append(
        cl.JaroWinklerAtThresholds(
            "nome_paciente_std",
            score_threshold_or_thresholds=list(cfg.name_thresholds),
        )
    )

    # Primeiro e último nome com ajuste por frequência.
    comparisons.append(
        cl.ExactMatch("primeiro_nome").configure(term_frequency_adjustments=True)
    )
    comparisons.append(
        cl.ExactMatch("ultimo_nome").configure(term_frequency_adjustments=True)
    )

    if cfg.include_dob:
        comparisons.append(
            cl.DateOfBirthComparison(
                "dt_nasc_std",
                input_is_string=True,
                datetime_metrics=["day", "month", "year"],
                datetime_thresholds=[1, 1, 1],
            )
        )

    if cfg.include_sex:
        comparisons.append(
            cl.ExactMatch("sexo_std").configure(term_frequency_adjustments=True)
        )

    if cfg.include_ids:
        comparisons.append(
            cl.ExactMatch("cpf_digits").configure(term_frequency_adjustments=True)
        )
        comparisons.append(
            cl.ExactMatch("cns_digits").configure(term_frequency_adjustments=True)
        )

    if cfg.include_parents:
        comparisons.append(
            cl.JaroWinklerAtThresholds(
                "nome_mae_std",
                score_threshold_or_thresholds=list(cfg.parent_thresholds),
            )
        )
        comparisons.append(
            cl.JaroWinklerAtThresholds(
                "nome_pai_std",
                score_threshold_or_thresholds=list(cfg.parent_thresholds),
            )
        )

    return comparisons


def montar_settings(cfg: ExperimentConfig) -> SettingsCreator:
    return SettingsCreator(
        link_type="dedupe_only",
        unique_id_column_name="unique_id",
        source_dataset_column_name="source_dataset",
        retain_matching_columns=True,
        retain_intermediate_calculation_columns=True,
        additional_columns_to_retain=["id_unico", "fonte_dados"],
        blocking_rules_to_generate_predictions=regras_blocking(cfg.blocking_scenario),
        comparisons=montar_comparisons(cfg),
    )


def gerar_grid_experimentos(quick: bool = True) -> List[ExperimentConfig]:
    """
    quick=True: grid reduzido para validar se tudo roda.
    quick=False: bateria mais ampla para stress test.
    """
    blocking_scenarios = [
        "ids_strict",
        "ids_plus_nome_data",
        "balanced",
        "no_ids_moderado",
        "broad_recall",
    ]

    comparison_scenarios = [
        {
            "comparison_scenario": "completo",
            "include_ids": True,
            "include_parents": True,
            "include_dob": True,
            "include_sex": True,
        },
        {
            "comparison_scenario": "sem_ids",
            "include_ids": False,
            "include_parents": True,
            "include_dob": True,
            "include_sex": True,
        },
        {
            "comparison_scenario": "sem_pais",
            "include_ids": True,
            "include_parents": False,
            "include_dob": True,
            "include_sex": True,
        },
        {
            "comparison_scenario": "nome_data_sexo",
            "include_ids": False,
            "include_parents": False,
            "include_dob": True,
            "include_sex": True,
        },
    ]

    name_thresholds_grid = [
        (0.98, 0.95, 0.90),
        (0.95, 0.90, 0.85),
        (0.92, 0.88, 0.80),
    ]

    parent_thresholds_grid = [
        (0.95, 0.90, 0.85),
        (0.92, 0.88, 0.80),
    ]

    training_modes = [
        "em_unsupervised",
        "labels_upper_bound",
    ]

    prior_recalls = [0.70, 0.85, 0.95]
    u_max_pairs_grid = [1e6, 5e6]
    prediction_min_thresholds = [0.01]

    if quick:
        blocking_scenarios = ["ids_plus_nome_data", "balanced", "no_ids_moderado"]
        comparison_scenarios = comparison_scenarios[:2]
        name_thresholds_grid = [(0.95, 0.90, 0.85)]
        parent_thresholds_grid = [(0.92, 0.88, 0.80)]
        training_modes = ["em_unsupervised", "labels_upper_bound"]
        prior_recalls = [0.85]
        u_max_pairs_grid = [1e6]

    configs = []

    for block, comp, nt, pt, train, recall, u_pairs, pred_thr in product(
        blocking_scenarios,
        comparison_scenarios,
        name_thresholds_grid,
        parent_thresholds_grid,
        training_modes,
        prior_recalls,
        u_max_pairs_grid,
        prediction_min_thresholds,
    ):
        # Evita combinação incoerente: comparação sem IDs com blocking só por IDs.
        if comp["include_ids"] is False and block == "ids_strict":
            continue

        nome = (
            f"block={block}__comp={comp['comparison_scenario']}__"
            f"nome={nt}__pais={pt}__train={train}__recall={recall}__u={int(u_pairs)}"
        )

        configs.append(
            ExperimentConfig(
                nome=nome,
                blocking_scenario=block,
                comparison_scenario=comp["comparison_scenario"],
                name_thresholds=nt,
                parent_thresholds=pt,
                include_ids=comp["include_ids"],
                include_parents=comp["include_parents"],
                include_dob=comp["include_dob"],
                include_sex=comp["include_sex"],
                training_mode=train,
                prior_recall=recall,
                u_max_pairs=u_pairs,
                prediction_min_threshold=pred_thr,
            )
        )

    return configs


def safe_filename(s: str, max_len: int = 140) -> str:
    s = re.sub(r"[^a-zA-Z0-9_.=-]+", "_", s)
    return s[:max_len]


# 5. Treinamento e avaliação



Esta célula contém a lógica de execução dos experimentos.



### O que o Splink precisa estimar



O Splink usa uma lógica probabilística. De forma simples, ele tenta estimar:

- `u`: chance de dois registros diferentes coincidirem em determinado campo por acaso.
- `m`: chance de dois registros da mesma pessoa coincidirem em determinado campo.



### Dois modos de treino usados


* Não supervisionado: O algoritmo tenta estimar os parâmetros sem usar a verdade conhecida diretamente. Isso simula um cenário mais realista, em que você não tem uma coluna confiável de verdade.

* Labels upper bound: Usa `id_unico` para estimar os parâmetros `m`. Este cenário funciona como referência de teto, porque estamos usando uma verdade conhecida para calibrar o modelo.




### Métricas



Para cada cenário, o notebook tenta calcular:



| Métrica | O que responde |
|---|---|
| Precision | Dos pares que o modelo disse que eram match, quantos eram verdadeiros? |
| Recall | Dos pares verdadeiros, quantos o modelo conseguiu encontrar? |
| F1 | Equilíbrio entre precision e recall |
| Specificity | Capacidade de evitar falso pareamento |
| Accuracy | Acerto geral, mas pode enganar em dados desbalanceados |



Normalmente, falso pareamento é sensível. Juntar duas pessoas diferentes pode ser mais grave do que deixar um par verdadeiro sem unir. Por isso, optamos por F1 e depois também precision e falsos positivos.

### Funções de treinamento e avaliação

In [5]:

# ============================================================
# Avaliação, execução dos experimentos e ranking
# ============================================================

def total_pares_verdadeiros(df: pd.DataFrame) -> int:
    vc = df["id_unico"].value_counts()
    return int(((vc * (vc - 1)) // 2).sum())


def total_pares_possiveis(df: pd.DataFrame) -> int:
    n = len(df)
    return int(n * (n - 1) // 2)


def extrair_melhor_linha_metricas(roc_df: pd.DataFrame) -> Dict[str, Any]:
    cols_lower = {c.lower(): c for c in roc_df.columns}

    f1_col = None
    for candidato in ["f1", "f1_score", "f1 score"]:
        if candidato in cols_lower:
            f1_col = cols_lower[candidato]
            break

    if f1_col is None:
        possiveis = [c for c in roc_df.columns if "f1" in c.lower()]
        f1_col = possiveis[0] if possiveis else None

    if f1_col is None:
        precision_col = next((c for c in roc_df.columns if "precision" in c.lower()), None)
        recall_col = next((c for c in roc_df.columns if "recall" in c.lower()), None)

        if precision_col and recall_col:
            p = pd.to_numeric(roc_df[precision_col], errors="coerce")
            r = pd.to_numeric(roc_df[recall_col], errors="coerce")
            roc_df = roc_df.copy()
            roc_df["f1_calculado"] = np.where((p + r) > 0, 2 * p * r / (p + r), np.nan)
            f1_col = "f1_calculado"
        else:
            return {"erro_metricas": "Não encontrei coluna de F1, precision ou recall."}

    idx = pd.to_numeric(roc_df[f1_col], errors="coerce").idxmax()
    row = roc_df.loc[idx].to_dict()

    saida = {
        "best_metric_col": f1_col,
        "best_f1": row.get(f1_col),
    }

    for c in roc_df.columns:
        clower = c.lower()
        if any(k in clower for k in [
            "precision", "recall", "specificity", "accuracy", "threshold",
            "match_weight", "match_probability", "false", "true", "f1",
            "phi", "npv"
        ]):
            saida[f"best_{c}"] = row.get(c)

    # Padroniza nomes mais comuns para facilitar ranking.
    for k in list(saida.keys()):
        kl = k.lower()
        if "precision" in kl and "best_precision" not in saida:
            saida["best_precision"] = saida[k]
        if "recall" in kl and "best_recall" not in saida:
            saida["best_recall"] = saida[k]

    return saida


def metricas_manuais_de_predicoes(
    pred_df: pd.DataFrame,
    df_input: pd.DataFrame,
    thresholds: List[float],
) -> pd.DataFrame:
    """
    Fallback caso a avaliação nativa do Splink falhe.
    """
    if "id_unico_l" not in pred_df.columns or "id_unico_r" not in pred_df.columns:
        raise ValueError(
            "As colunas id_unico_l/id_unico_r não vieram nas predições. "
            "Verifique additional_columns_to_retain e retain_matching_columns."
        )

    total_true = total_pares_verdadeiros(df_input)
    total_pairs = total_pares_possiveis(df_input)
    total_false = total_pairs - total_true

    pred_df = pred_df.copy()
    pred_df["is_true_match"] = pred_df["id_unico_l"].astype(str) == pred_df["id_unico_r"].astype(str)

    rows = []
    for thr in thresholds:
        p = pred_df[pred_df["match_probability"] >= thr]
        tp = int(p["is_true_match"].sum())
        fp = int((~p["is_true_match"]).sum())
        fn = int(total_true - tp)
        tn = int(total_false - fp)

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        specificity = tn / (tn + fp) if (tn + fp) else 0

        rows.append({
            "threshold_match_probability": thr,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "f1": f1,
            "n_predicted_matches": int(len(p)),
        })

    return pd.DataFrame(rows)


def estimar_custo_blocking(
    df: pd.DataFrame,
    cfg: ExperimentConfig,
    db_api: DuckDBAPI,
    max_rows_limit: int,
) -> Dict[str, Any]:
    """
    Estima custo de blocking quando a função está disponível.
    Se a versão instalada do Splink não tiver a função, pula essa etapa.
    """
    regras = regras_blocking(cfg.blocking_scenario)

    if not HAS_BLOCKING_ANALYSIS:
        return {
            "blocking_n_rules": len(regras),
            "blocking_upper_bound_comparisons": None,
            "blocking_details": [],
            "blocking_analysis_disponivel": False,
        }

    rows = []
    total_upper_bound = 0

    for i, br in enumerate(regras):
        try:
            info = count_comparisons_from_blocking_rule(
                table_or_tables=df,
                blocking_rule=br,
                link_type="dedupe_only",
                db_api=db_api,
                unique_id_column_name="unique_id",
                source_dataset_column_name="source_dataset",
                max_rows_limit=max_rows_limit,
            )

            n = int(info.get("number_of_comparisons_to_be_scored_post_filter_conditions", 0))
            total_upper_bound += n

            rows.append({
                "regra_idx": i,
                "blocking_rule": str(br),
                "n_comparisons_rule": n,
                "status": "ok",
            })

        except Exception as e:
            rows.append({
                "regra_idx": i,
                "blocking_rule": str(br),
                "n_comparisons_rule": None,
                "status": f"erro: {e}",
            })

    return {
        "blocking_n_rules": len(regras),
        "blocking_upper_bound_comparisons": total_upper_bound,
        "blocking_details": rows,
        "blocking_analysis_disponivel": True,
    }


def treinar_modelo(linker: Linker, cfg: ExperimentConfig) -> Dict[str, Any]:
    log = {
        "training_mode": cfg.training_mode,
        "training_errors": [],
    }

    try:
        linker.training.estimate_probability_two_random_records_match(
            regras_deterministicas_para_prior(),
            recall=cfg.prior_recall,
        )
        log["prior_estimado"] = True
    except Exception as e:
        log["prior_estimado"] = False
        log["training_errors"].append(f"prior: {e}")

    try:
        linker.training.estimate_u_using_random_sampling(
            max_pairs=cfg.u_max_pairs,
            seed=cfg.seed,
        )
        log["u_estimado"] = True
    except Exception as e:
        log["u_estimado"] = False
        log["training_errors"].append(f"u_sampling: {e}")

    if cfg.training_mode == "labels_upper_bound":
        try:
            linker.training.estimate_m_from_label_column("id_unico")
            log["m_estimado"] = "label_column"
        except Exception as e:
            log["m_estimado"] = "erro"
            log["training_errors"].append(f"m_label_column: {e}")

    elif cfg.training_mode == "em_unsupervised":
        em_ok = 0
        for br in regras_em_training():
            try:
                linker.training.estimate_parameters_using_expectation_maximisation(br)
                em_ok += 1
            except Exception as e:
                log["training_errors"].append(f"em {br}: {e}")

        log["m_estimado"] = f"em_sessions_ok={em_ok}"

    else:
        raise ValueError(f"training_mode inválido: {cfg.training_mode}")

    return log


def rodar_experimento(
    df: pd.DataFrame,
    cfg: ExperimentConfig,
    out_dir: Path,
    max_blocking_comparisons: int = 20_000_000,
    save_predictions: bool = False,
    save_charts: bool = False,
) -> Dict[str, Any]:
    t0 = time.time()
    out_dir.mkdir(parents=True, exist_ok=True)

    resultado = {
        **asdict(cfg),
        "status": "iniciado",
        "erro": None,
    }

    db_api = DuckDBAPI()

    try:
        custo = estimar_custo_blocking(
            df=df,
            cfg=cfg,
            db_api=db_api,
            max_rows_limit=max_blocking_comparisons,
        )

        resultado.update({
            "blocking_n_rules": custo["blocking_n_rules"],
            "blocking_upper_bound_comparisons": custo["blocking_upper_bound_comparisons"],
            "blocking_analysis_disponivel": custo.get("blocking_analysis_disponivel"),
        })

        details_path = out_dir / f"blocking_details__{safe_filename(cfg.nome)}.json"
        with open(details_path, "w", encoding="utf-8") as f:
            json.dump(custo["blocking_details"], f, ensure_ascii=False, indent=2)

        if (
            custo["blocking_upper_bound_comparisons"] is not None
            and custo["blocking_upper_bound_comparisons"] > max_blocking_comparisons
        ):
            resultado["status"] = "skipped_blocking_muito_grande"
            resultado["tempo_seg"] = round(time.time() - t0, 2)
            return resultado

        settings = montar_settings(cfg)
        linker = Linker(df, settings, db_api=db_api)

        log_treino = treinar_modelo(linker, cfg)
        resultado.update(log_treino)

        model_path = out_dir / f"model__{safe_filename(cfg.nome)}.json"
        try:
            linker.misc.save_model_to_json(model_path, overwrite=True)
            resultado["model_path"] = str(model_path)
        except Exception as e:
            resultado["model_path"] = None
            resultado["training_errors"] = resultado.get("training_errors", []) + [f"save_model: {e}"]

        df_pred = linker.inference.predict(
            threshold_match_probability=cfg.prediction_min_threshold
        )

        try:
            pred_pdf_min = df_pred.as_pandas_dataframe(limit=5)
            resultado["pred_sample_cols"] = list(pred_pdf_min.columns)
        except Exception:
            pass

        # Avaliação nativa com id_unico como verdade conhecida.
        roc_path = out_dir / f"accuracy_table__{safe_filename(cfg.nome)}.csv"

        try:
            roc = linker.evaluation.accuracy_analysis_from_labels_column(
                "id_unico",
                output_type="table",
                add_metrics=["f1", "specificity", "npv", "accuracy", "phi"],
                positives_not_captured_by_blocking_rules_scored_as_zero=True,
            )
            roc_df = roc.as_pandas_dataframe()
            roc_df.to_csv(roc_path, index=False)

            resultado.update(extrair_melhor_linha_metricas(roc_df))
            resultado["accuracy_table_path"] = str(roc_path)
            resultado["avaliacao"] = "splink_native"

        except Exception as e_native:
            # Fallback manual.
            try:
                pred_pdf = df_pred.as_pandas_dataframe()
                resultado["n_predicoes_min_threshold"] = len(pred_pdf)

                thresholds = [0.50, 0.70, 0.80, 0.90, 0.95, 0.97, 0.99]
                roc_df = metricas_manuais_de_predicoes(pred_pdf, df, thresholds)
                roc_df.to_csv(roc_path, index=False)

                resultado.update(extrair_melhor_linha_metricas(roc_df))
                resultado["accuracy_table_path"] = str(roc_path)
                resultado["avaliacao"] = "manual_fallback"
                resultado["erro_avaliacao_nativa"] = str(e_native)

            except Exception as e_fallback:
                resultado["avaliacao"] = "erro"
                resultado["erro"] = f"erro avaliação nativa: {e_native}; erro fallback: {e_fallback}"

        if save_predictions:
            try:
                pred_path = out_dir / f"predictions__{safe_filename(cfg.nome)}.parquet"
                pred_pdf = df_pred.as_pandas_dataframe()
                pred_pdf.to_parquet(pred_path, index=False)
                resultado["predictions_path"] = str(pred_path)
                resultado["n_predicoes_min_threshold"] = len(pred_pdf)
            except Exception as e:
                resultado["predictions_path"] = None
                resultado["prediction_save_error"] = str(e)

        if save_charts:
            try:
                chart_path = out_dir / f"comparison_viewer__{safe_filename(cfg.nome)}.html"
                linker.visualisations.comparison_viewer_dashboard(
                    df_pred,
                    str(chart_path),
                    overwrite=True,
                    num_example_rows=5,
                )
                resultado["comparison_viewer_path"] = str(chart_path)
            except Exception as e:
                resultado["comparison_viewer_path"] = None
                resultado["chart_error"] = str(e)

        resultado["status"] = "ok" if resultado.get("erro") is None else "ok_com_alerta"
        resultado["tempo_seg"] = round(time.time() - t0, 2)
        return resultado

    except Exception as e:
        resultado["status"] = "erro"
        resultado["erro"] = str(e)
        resultado["tempo_seg"] = round(time.time() - t0, 2)
        return resultado


def executar_stress_test(
    input_csv: str | Path,
    out_dir: str | Path = "/content/saida_splink",
    sample_records: int = 50_000,
    quick: bool = True,
    max_blocking_comparisons: int = 20_000_000,
    run_full_best: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Função principal do notebook.

    Retorna:
    - df_full preparado
    - df_sample preparado
    - df_resultados com ranking dos cenários
    """
    input_csv = Path(input_csv)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Lendo e preparando dados...")
    df_full = preparar_df(input_csv)

    perfil_full = perfil_basico(df_full)
    with open(out_dir / "perfil_full.json", "w", encoding="utf-8") as f:
        json.dump(perfil_full, f, ensure_ascii=False, indent=2)

    print("Perfil do arquivo completo:")
    display(mostrar_perfil(perfil_full))

    print("\nCriando amostra preservando clusters por id_unico...")
    df_sample = amostrar_por_pessoa(df_full, sample_records, seed=42)

    perfil_sample = perfil_basico(df_sample)
    with open(out_dir / "perfil_sample.json", "w", encoding="utf-8") as f:
        json.dump(perfil_sample, f, ensure_ascii=False, indent=2)

    print("Perfil da amostra:")
    display(mostrar_perfil(perfil_sample))

    df_sample.to_csv(out_dir / "df_splink_preparado_sample.csv", index=False)

    configs = gerar_grid_experimentos(quick=quick)
    print(f"\nTotal de cenários a testar: {len(configs)}")

    resultados = []
    resultados_path = out_dir / "resultados_splink_stress_test.csv"

    for i, cfg in enumerate(configs, start=1):
        print(f"\n[{i}/{len(configs)}] {cfg.nome}")

        res = rodar_experimento(
            df=df_sample,
            cfg=cfg,
            out_dir=out_dir / "experimentos",
            max_blocking_comparisons=max_blocking_comparisons,
            save_predictions=False,
            save_charts=False,
        )

        resultados.append(res)
        pd.DataFrame(resultados).to_csv(resultados_path, index=False)

        print({
            "status": res.get("status"),
            "best_f1": res.get("best_f1"),
            "best_precision": res.get("best_precision"),
            "best_recall": res.get("best_recall"),
            "tempo_seg": res.get("tempo_seg"),
            "erro": res.get("erro"),
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(resultados_path, index=False)

    print("\nRanking dos melhores cenários:")
    cols_show = [
        "status", "best_f1", "best_precision", "best_recall",
        "blocking_scenario", "comparison_scenario", "training_mode",
        "name_thresholds", "parent_thresholds", "prior_recall", "u_max_pairs",
        "tempo_seg", "erro"
    ]
    cols_show = [c for c in cols_show if c in df_resultados.columns]

    display(
        df_resultados
        .assign(best_f1_num=lambda d: pd.to_numeric(d["best_f1"], errors="coerce") if "best_f1" in d else np.nan)
        .sort_values("best_f1_num", ascending=False, na_position="last")[cols_show]
        .head(30)
    )

    if run_full_best:
        ok = df_resultados[df_resultados["status"].isin(["ok", "ok_com_alerta"])].copy()
        ok["best_f1_num"] = pd.to_numeric(ok["best_f1"], errors="coerce")
        ok = ok.sort_values("best_f1_num", ascending=False)

        if len(ok) == 0:
            print("Nenhum cenário válido para rodar no arquivo completo.")
            return df_full, df_sample, df_resultados

        best_name = ok.iloc[0]["nome"]
        best_cfg = next(c for c in configs if c.nome == best_name)

        print("\nRodando melhor cenário no arquivo completo:")
        print(best_cfg)

        res_full = rodar_experimento(
            df=df_full,
            cfg=best_cfg,
            out_dir=out_dir / "melhor_cenario_full",
            max_blocking_comparisons=max_blocking_comparisons,
            save_predictions=True,
            save_charts=True,
        )

        with open(out_dir / "resultado_melhor_cenario_full.json", "w", encoding="utf-8") as f:
            json.dump(res_full, f, ensure_ascii=False, indent=2)

        print("Resultado do melhor cenário no arquivo completo:")
        print(json.dumps(res_full, ensure_ascii=False, indent=2))

    return df_full, df_sample, df_resultados


# 6. Teste rápido



Esse teste roda uma bateria reduzida em aproximadamente 50 mil registros.

O objetivo não é encontrar o resultado definitivo. O objetivo é validar:

- se o CSV foi lido corretamente;
- se as colunas foram reconhecidas;
- se o Splink está instalado;
- se as regras não estão gerando erro;
- se o tempo de execução está aceitável;
- se as métricas aparecem no final.



In [8]:
df_full, df_sample, resultados_quick = executar_stress_test(
    input_csv=csv_path,
    out_dir=out_dir,
    sample_records=50_000,
    quick=True,
    max_blocking_comparisons=20_000_000,
    run_full_best=False,
)


Lendo e preparando dados...
Perfil do arquivo completo:


,indicador,valor
0,n_registros,2.000000e+05
1,n_pessoas_reais,1.200000e+05
2,n_pessoas_com_duplicidade,5.832300e+04
3,maior_cluster_real,8.000000e+00
4,total_pares_possiveis,1.999990e+10
5,total_pares_verdadeiros,1.067570e+05
6,proporcao_pares_verdadeiros,0.000000e+00
7,pares_verdadeiros_interfonte,8.536200e+04
8,pares_verdadeiros_intrafonte,2.139500e+04
9,perc_pares_verdadeiros_interfonte,7.996000e-01



Criando amostra preservando clusters por id_unico...
Perfil da amostra:


,indicador,valor
0,n_registros,5.000000e+04
1,n_pessoas_reais,3.000700e+04
2,n_pessoas_com_duplicidade,1.460500e+04
3,maior_cluster_real,7.000000e+00
4,total_pares_possiveis,1.249975e+09
5,total_pares_verdadeiros,2.657700e+04
6,proporcao_pares_verdadeiros,0.000000e+00
7,pares_verdadeiros_interfonte,2.117800e+04
8,pares_verdadeiros_intrafonte,5.399000e+03
9,perc_pares_verdadeiros_interfonte,7.969000e-01



Total de cenários a testar: 12

[1/12] block=ids_plus_nome_data__comp=completo__nome=(0.95, 0.9, 0.85)__pais=(0.92, 0.88, 0.8)__train=em_unsupervised__recall=0.85__u=1000000


...

INFO:splink.internals.m_u_records_to_parameters:u probability not trained for nome_mae_std - Exact match on nome_mae_std (comparison vector value: 4). This usually means the comparison level was never observed in the training data.
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - nome_paciente_std (no m values are trained).
    - primeiro_nome (no m values are trained).
    - ultimo_nome (no m values are trained).
    - dt_nasc_std (no m values are trained).
    - sexo_std (no m values are trained).
    - nome_mae_std (some u values are not trained, no m values are trained).
    - nome_pai_std (no m values are trained).
INFO:splink.internals.m_training:------- Estimating m probabilities using from column id_unico --------
INFO:splink.internals.m_u_records_to_parameters:m probability not trained for nome_paciente_std - All other comparisons (comparison vecto

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

INFO:splink.internals.linker_components.inference:Predict time: 2.56 seconds
 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'nome_paciente_std':
    m values not fully trained
Comparison: 'dt_nasc_std':
    m values not fully trained
Comparison: 'sexo_std':
    m values not fully trained
Comparison: 'nome_mae_std':
    m values not fully trained
Comparison: 'nome_mae_std':
    u values not fully trained
Comparison: 'nome_pai_std':
    m values not fully trained


{'status': 'ok', 'best_f1': 0.9626764122701295, 'best_precision': 0.9958174140357933, 'best_recall': 0.9316702411859878, 'tempo_seg': 17.34, 'erro': None}

Ranking dos melhores cenários:


,status,best_f1,best_precision,best_recall,blocking_scenario,comparison_scenario,training_mode,name_thresholds,parent_thresholds,prior_recall,u_max_pairs,tempo_seg,erro
4,ok,0.997947,0.998907,0.996990,balanced,completo,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,129.20,None
5,ok,0.997120,0.997515,0.996726,balanced,completo,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,16.88,None
0,ok,0.984642,1.000000,0.969748,ids_plus_nome_data,completo,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,140.69,None
1,ok,0.984622,1.000000,0.969711,ids_plus_nome_data,completo,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,16.29,None
3,ok,0.984433,0.999728,0.969598,ids_plus_nome_data,sem_ids,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,12.88,None
2,ok,0.984430,0.999884,0.969447,ids_plus_nome_data,sem_ids,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,331.91,None
8,ok,0.971204,0.999841,0.944162,no_ids_moderado,completo,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,329.39,None
9,ok,0.971164,0.999841,0.944087,no_ids_moderado,completo,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,20.76,None
7,ok,0.964528,0.995833,0.935132,balanced,sem_ids,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,16.32,None
11,ok,0.962676,0.995817,0.931670,no_ids_moderado,sem_ids,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,17.34,None


### Ranking dos cenários



Esta célula organiza os resultados do teste rápido.

A leitura principal é:

- `best_f1`: melhor equilíbrio geral.
- `best_precision`: segurança contra falsos positivos.
- `best_recall`: capacidade de recuperar pares verdadeiros.
- `blocking_scenario`: regra de geração de candidatos.
- `comparison_scenario`: variáveis usadas na comparação.
- `training_mode`: forma de treino.
- `tempo_seg`: tempo de execução do cenário.



Um cenário com F1 alto, mas precision baixa, pode estar juntando pessoas erradas.

Um cenário com precision alta, mas recall baixo, pode estar sendo conservador demais.

Para uso em saúde, olhar primeiro:

1. precision;
2. recall;
3. F1;
4. custo computacional;
5. dependência de CPF/CNS.

In [9]:
cols = [c for c in [
    'status', 'best_f1', 'best_precision', 'best_recall',
    'blocking_scenario', 'comparison_scenario', 'training_mode',
    'name_thresholds', 'parent_thresholds', 'prior_recall', 'u_max_pairs',
    'tempo_seg', 'erro'
] if c in resultados_quick.columns]

ranking_quick = (
    resultados_quick
    .assign(best_f1_num=lambda d: pd.to_numeric(d['best_f1'], errors='coerce') if 'best_f1' in d else np.nan)
    .sort_values('best_f1_num', ascending=False, na_position='last')
)

ranking_quick[cols].head(30)

,status,best_f1,best_precision,best_recall,blocking_scenario,comparison_scenario,training_mode,name_thresholds,parent_thresholds,prior_recall,u_max_pairs,tempo_seg,erro
4,ok,0.997947,0.998907,0.996990,balanced,completo,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,129.20,None
5,ok,0.997120,0.997515,0.996726,balanced,completo,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,16.88,None
0,ok,0.984642,1.000000,0.969748,ids_plus_nome_data,completo,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,140.69,None
1,ok,0.984622,1.000000,0.969711,ids_plus_nome_data,completo,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,16.29,None
3,ok,0.984433,0.999728,0.969598,ids_plus_nome_data,sem_ids,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,12.88,None
2,ok,0.984430,0.999884,0.969447,ids_plus_nome_data,sem_ids,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,331.91,None
8,ok,0.971204,0.999841,0.944162,no_ids_moderado,completo,em_unsupervised,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,329.39,None
9,ok,0.971164,0.999841,0.944087,no_ids_moderado,completo,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,20.76,None
7,ok,0.964528,0.995833,0.935132,balanced,sem_ids,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,16.32,None
11,ok,0.962676,0.995817,0.931670,no_ids_moderado,sem_ids,labels_upper_bound,"(0.95, 0.9, 0.85)","(0.92, 0.88, 0.8)",0.85,1000000.0,17.34,None


Conclusão:
* O notebook está funcionando corretamente.
* O dado tem sinal forte para pareamento.
* O melhor cenário foi balanced + completo.
* CPF/CNS ajudam, mas o modelo ainda funciona razoavelmente sem eles.
* em_unsupervised: mais realista, porque treina sem usar diretamente o id_unico.
* labels_upper_bound: usa id_unico para ajudar no treino. Serve como referência ou cenário de teto.
* como o labels_upper_bound foi muito mais rápido e quase tão bom, ele é ótimo para benchmarking.
* para simular um uso real sem verdade conhecida, o EM é mais honesto.
* o cenário balanced funcionou melhor porque combina CPF/CNS, nome, data de nascimento e nome da mãe no blocking.

# 6. Teste final

Esta etapa roda mais cenários e usa uma amostra maior. Ela tende a demorar mais, porque combina mais opções:

- mais cenários de blocking;
- mais limiares de similaridade;
- mais estratégias de treino;
- mais hipóteses sobre uso ou não de identificadores.

In [ ]:
df_full, df_sample, resultados_stress = executar_stress_test(
    input_csv=csv_path,
    out_dir=out_dir,
    sample_records=80_000,
    quick=False,
    max_blocking_comparisons=20_000_000,
    run_full_best=False,
)


### Ranking dos cenários


Depois de encontrar bons candidatos na bateria mais forte, esta etapa roda o melhor cenário no dataset completo.

Aqui o objetivo é sair da simulação em amostra e avaliar a configuração escolhida em todos os registros.

Esta etapa pode demorar mais. Ela também pode gerar muitos pares candidatos se o melhor cenário tiver blocking muito permissivo.

Se der erro de memória ou tempo, faça uma das opções:
- aumente a restrição do blocking;
- reduza `max_blocking_comparisons`;
- rode apenas os 2 ou 3 cenários candidatos manualmente;
- use uma máquina com mais memória.

In [ ]:
df_full, df_sample, resultados_final = executar_stress_test(
    input_csv=csv_path,
    out_dir=out_dir,
    sample_records=80_000,
    quick=False,
    max_blocking_comparisons=20_000_000,
    run_full_best=True,
)


# 7. Teste final

Esta célula compacta a pasta `saida_splink` em um arquivo `.zip`.

Dentro da pasta, os principais arquivos são:

| Arquivo | Conteúdo |
|---|---|
| `perfil_full.json` | Perfil do dataset completo |
| `perfil_sample.json` | Perfil da amostra |
| `resultados_splink_stress_test.csv` | Ranking dos cenários |
| `accuracy_table__*.csv` | Métricas por limiar em cada cenário |
| `model__*.json` | Modelo Splink salvo |
| `predictions__*.parquet` | Pares preditos no melhor cenário completo, quando executado |

Guarde esse `.zip`, porque ele permite revisar a escolha do modelo depois.

In [ ]:
!zip -r /content/saida_splink.zip /content/saida_splink
files.download('/content/saida_splink.zip')


# 8. Experimentos manuais recomendados



Depois que o notebook rodar, vale testar manualmente alguns cenários específicos.


**Cenário A: alta segurança**



Use quando falso pareamento for muito perigoso.

- blocking: `ids_plus_nome_data`
- comparação: `completo`
- nome: limiares mais rígidos
- foco: precision


**Cenário B: sem depender de CPF/CNS**


Use para simular bases com identificadores ruins.

- blocking: `no_ids_moderado`
- comparação: `sem_ids`
- foco: recall e revisão de falsos positivos


**Cenário C: maior recuperação**



Use para descobrir pares que regras rígidas deixam passar.

- blocking: `broad_recall`
- comparação: `completo`
- foco: recall, custo computacional e revisão manual


**Cenário D: modelo simples e explicável**



Use quando você precisa explicar facilmente para gestores ou equipe técnica.

- blocking: `ids_plus_nome_data`
- comparação: `nome_data_sexo`
- foco: simplicidade, tempo e interpretabilidade

# 9. Considerações para decisão

Não escolha apenas pelo maior F1. Para um projeto real, recomendação de seguir esta ordem:


**1. Remova cenários inviáveis**

Exclua cenários que:

- demoraram demais;
- geraram erro;
- dependeram demais de campos muito incompletos;
- tiveram blocking muito amplo.

**2. Olhe F1**



Use como métrica de equilíbrio, não como decisão automática.

**3. Olhe precision**

Se o cenário junta pessoas diferentes, isso é perigoso.

Em saúde, falso positivo de pareamento pode contaminar análise, prontuário, linkage e indicadores.

**4. Olhe recall**



Se o cenário perde muitos pares verdadeiros, você terá fragmentação.

A mesma pessoa pode aparecer como várias pessoas diferentes.

**5. Faça revisão amostral**



Pegue uma amostra de:

- pares com score alto;
- pares próximos ao limiar;
- pares falsos positivos;
- pares falsos negativos.

Isso ajuda a entender se o erro vem de nome, data, CPF, CNS, mãe, pai ou fonte.

**6. Defina dois limiares, se necessário**




Em vez de um único corte, você pode usar três faixas:

| Faixa | Ação |
|---|---|
| Score alto | Parear automaticamente |
| Score intermediário | Revisão manual |
| Score baixo | Não parear |

Essa abordagem costuma ser mais segura para dados sensíveis.

In [ ]:
# ============================================================
# Célula opcional: leitura do ranking salvo
# ============================================================

import pandas as pd
from pathlib import Path

ranking_path = Path(out_dir) / "resultados_splink_stress_test.csv"

if ranking_path.exists():
    df_ranking = pd.read_csv(ranking_path)

    cols = [c for c in [
        "status", "best_f1", "best_precision", "best_recall",
        "blocking_scenario", "comparison_scenario", "training_mode",
        "name_thresholds", "parent_thresholds", "prior_recall",
        "u_max_pairs", "tempo_seg", "erro"
    ] if c in df_ranking.columns]

    df_ranking = (
        df_ranking
        .assign(best_f1_num=lambda d: pd.to_numeric(d["best_f1"], errors="coerce") if "best_f1" in d else None)
        .sort_values("best_f1_num", ascending=False, na_position="last")
    )

    display(df_ranking[cols].head(50))
else:
    print("Ainda não encontrei o arquivo de ranking. Rode primeiro o teste rápido ou a bateria forte.")